An exploration where we test different models and tune them in order to find the best one that we finally use in this project.

We'll test logistic regression, random forest and XGBoost.
My initial guess is that Random forest will perform best since the data is quite scarce for an XGBoost to be optimal and it's correlations seems non linear which is to logistic regressions disadvantage. 

In [4]:
# Libraries:
import pandas as pd
import sqlite3
import os
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score

In [ ]:
# path to database
# Adjust the path if your notebook is in the 'Notebooks' folder
db_path = os.path.join('..', 'Database', 'churn_database.db')

# querying the cleaned data
conn = sqlite3.connect(db_path)
df_cleaned = pd.read_sql("SELECT * FROM prd_churn_cleaned", conn)
conn.close()

# defining Features (X) and Target (y)
X = df_cleaned.drop('Exited', axis=1)
y = df_cleaned['Exited']

# splitting the data 80/20 (train/test)
# I use stratify=y to keep the 80/20 churn ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"training rows: {len(X_train)}, Testing rows: {len(X_test)}")

training rows: 8000, Testing rows: 2000


In [5]:
# models we'll test
base_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

# looping though each model and printing their scores
results = []
for name, model in base_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "F1": f1_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_proba)
    })

# creating dataFrame for viewing
comparison_df = pd.DataFrame(results)
print(comparison_df)

/home/arash/Desktop/churn_pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


                 Model        F1       AUC
0  Logistic Regression  0.317518  0.759379
1        Random Forest  0.602740  0.862944
2              XGBoost  0.589888  0.850972


As believed, RF and XGBoost did way better on this data than the logistic regression model. moving forward, we will tune RF and XGBoost to see if we get a difference. 

In [6]:
from sklearn.model_selection import RandomizedSearchCV

rf_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced', 'balanced_subsample', None]
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42), 
    param_distributions=rf_grid, 
    n_iter=10, cv=3, scoring='f1', random_state=42, n_jobs=-1
)

rf_search.fit(X_train, y_train)
print(f"Best RF F1: {rf_search.best_score_:.4f}")
print(f"Best RF Params: {rf_search.best_params_}")

Best RF F1: 0.6143
Best RF Params: {'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 10, 'class_weight': 'balanced'}


We see F1 slightly raised for RF

In [9]:
# Modern XGBoost GPU syntax (v2.0+)
xgb_gpu = XGBClassifier(
    device="cuda", 
    eval_metric='logloss', 
    random_state=42
)

# running a single fit first just to see if 4070 "wakes up"
try:
    xgb_gpu.fit(X_train, y_train)
    print("✅ GPU Acceleration is working! 4070 is online.")
except Exception as e:
    print(f"❌ GPU Error: {e}")
    print("Falling back to CPU for now...")

✅ GPU Acceleration is working! 4070 is online.


In [11]:
xgb_grid = {
    'n_estimators': [100, 200, 500], # doing plenty estimators since i'm running on GPU
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 8],
    'scale_pos_weight': [1, 4],
    # GPU specific parameters
    'tree_method': ['gpu_hist'], 
    'predictor': ['gpu_predictor'],
    'device': ['cuda'] # For newer versions of XGBoost
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42), 
    param_distributions=xgb_grid, 
    n_iter=20,
    cv=3, 
    scoring='f1', 
    random_state=42, 
    n_jobs=1 # Keep this at 1 when using GPU to avoid memory conflicts
)